## Analysis of Bayesian Factorization Machine Uncertainty

This notebook reads the out-of-fold (OOF) predictions from a trained Bayesian Factorization Machine (BFM) to perform a detailed uncertainty analysis. The training script was specifically designed to separate the two primary components of predictive uncertainty:

1.  **Epistemic Uncertainty** (or *Model Uncertainty*): This represents the model's uncertainty about its own parameters. It captures what the model *doesn't know* and can be reduced by providing more training data.
2.  **Aleatoric Uncertainty** (or *Data Uncertainty*): This represents the inherent, irreducible noise or randomness in the data itself (e.g., measurement error, biological variability). It cannot be reduced by collecting more data of the same kind.

Our goal is to quantify these uncertainties globally and then analyze how they behave as a function of the number of observations available for a given chemical.

In [ ]:
import os
os.chdir('/home/tad/Desktop/Thesisfiles/ThesisCode/ecotox-toolkit/')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Use a clean and professional plot style
plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Load Data and OOF Predictions ---
print("Loading data and prediction artifacts...")
ART = Path("artifacts_final")
DATA_DIR = Path("/home/tad/Desktop/Thesisfiles/ThesisCode/ecotox-toolkit/data_files")

# Reload the data in the exact same order used for training
# This import requires the dataloaders module to be accessible
from dataloaders.load_ecotox import load_ecotox_data
full_data, y_centered = load_ecotox_data(
    adore_path=DATA_DIR / "ecotox_mortality_processed.csv",
    chemicals_path=DATA_DIR / "ecotox_properties_with-oecd-function.csv",
    shuffle=True,
    random_state=42,
    use_selfies=False, use_mol2vec=False, use_fingerprint=False,
)

# Load the pre-computed OOF arrays
oof_mean = np.load(ART / "oof_mean_64.npy")
oof_epistemic_var = np.load(ART / "oof_epistemic_var_64.npy")
oof_aleatoric_var = np.load(ART / "oof_aleatoric_var_64.npy")

assert len(full_data) == oof_mean.size, "Data and prediction length mismatch!"

print(f"Loaded {len(full_data)} data points and corresponding predictions.")

In [ ]:
# --- 2. Attach Predictions and Calculate Standard Deviations ---

# Create a fresh copy to avoid SettingWithCopyWarning
df = full_data.copy()

# Attach the mean prediction and the two sources of variance
df["pred_mean"] = oof_mean
df["epistemic_var"] = oof_epistemic_var
df["aleatoric_var"] = oof_aleatoric_var

# The total predictive variance is the sum of its components
df["total_var"] = df["epistemic_var"] + df["aleatoric_var"]

# Calculate standard deviations for easier interpretation (same units as target)
df["epistemic_sd"] = np.sqrt(df["epistemic_var"])
df["aleatoric_sd"] = np.sqrt(df["aleatoric_var"])
df["total_sd"] = np.sqrt(df["total_var"])

print("Uncertainty components attached to the dataframe:")
display(df[['CAS', 'species', 'pred_mean', 'epistemic_sd', 'aleatoric_sd', 'total_sd']].head())

### Mathematical Background: The Law of Total Variance

The total predictive variance for a new data point $x_*$ given the training data $D$ is formally expressed by the Law of Total Variance. Let $\theta$ represent the model parameters (`w0`, `w`, `v`, `alpha`):

$$ \text{Var}(y_* | D) = \mathbb{E}_{p(\theta|D)}[\text{Var}(y_*|\theta)] + \text{Var}_{p(\theta|D)}[\mathbb{E}(y_*|\theta)] $$

This decomposes into:

$$ \underbrace{\text{Total Variance}}_{\texttt{total\_var}} = \underbrace{\text{Aleatoric Variance}}_{\texttt{aleatoric\_var}} + \underbrace{\text{Epistemic Variance}}_{\texttt{epistemic\_var}} $$

- **Aleatoric Variance**: `E[Var(y|θ)]`, is the expected value of the data noise variance (`1/alpha`) over the posterior. This is the noise we expect on average.
- **Epistemic Variance**: `Var(E[y|θ])`, is the variance of the mean prediction as we vary the parameters according to the posterior. This is precisely what we saved as `oof_epistemic_var`.

In [ ]:
# --- 3. Global Average Uncertainty Metrics ---

# Calculate the average of each variance component across all predictions
avg_epistemic_var = df["epistemic_var"].mean()
avg_aleatoric_var = df["aleatoric_var"].mean()
avg_total_var = df["total_var"].mean()

print("="*50)
print("Global Average Uncertainty Analysis")
print("="*50)
print(f"Average Epistemic Variance (Model Uncertainty):   {avg_epistemic_var:.4f}")
print(f"Average Aleatoric Variance (Data Noise):          {avg_aleatoric_var:.4f}")
print(f"Average Total Predictive Variance:                {avg_total_var:.4f}")
print("-"*50)
# Also show the square root for interpretability
print(f"Avg. Model Standard Deviation:      {np.sqrt(avg_epistemic_var):.4f}")
print(f"Avg. Data Noise Standard Deviation:   {np.sqrt(avg_aleatoric_var):.4f}")
print(f"Avg. Total Standard Deviation:      {np.sqrt(avg_total_var):.4f}")
print("="*50)

## Uncertainty vs. Number of Observations

Now, we investigate the core question: how does the amount of data we have for a chemical affect the model's uncertainty? We will group the dataset by chemical (`CAS`) and plot the number of observations against each of the three types of uncertainty.

In [ ]:
# --- 4. Group by Chemical to Analyze Per-Chemical Uncertainty ---

chem_summary = (
    df.groupby("CAS")
    .agg(
        n_obs=("CAS", "size"),
        avg_epistemic_sd=("epistemic_sd", "mean"),
        avg_aleatoric_sd=("aleatoric_sd", "mean"),
        avg_total_sd=("total_sd", "mean"),
    )
    .sort_values("avg_total_sd", ascending=True)
)

chem_summary[chem_summary['n_obs']>10]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_uncertainty_vs_obs(data, y_col, title):
    """Create a single scatter plot of uncertainty vs. n_obs."""
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.scatterplot(
        data=data,
        x="n_obs",
        y=y_col,
        alpha=0.6,
        ax=ax,
        edgecolor="white",
        linewidth=0.5
    )
    ax.set_xscale("log")
    ax.set_title(title, fontsize=16, pad=15)
    ax.set_xlabel("Number of Observations (log scale)", fontsize=12)
    ax.set_ylabel("Average Standard Deviation", fontsize=12)
    fig.tight_layout()
    return fig, ax

# --- Plot 1: Epistemic Uncertainty ---
plot_uncertainty_vs_obs(
    chem_summary, "avg_epistemic_sd",
    "Model (Epistemic) Uncertainty vs. Observations"
)

# --- Plot 2: Aleatoric Uncertainty ---
plot_uncertainty_vs_obs(
    chem_summary, "avg_aleatoric_sd",
    "Data (Aleatoric) Uncertainty vs. Observations"
)

# --- Plot 3: Total Uncertainty ---
plot_uncertainty_vs_obs(
    chem_summary, "avg_total_sd",
    "Total Predictive Uncertainty vs. Observations"
)

plt.show()  # optional; in notebooks, each figure will usually display automatically


In [ ]:
# ==== SSD with uncertainty bands for a selected chemical (fixed cell) ====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def prepare_species_values(
    df: pd.DataFrame,
    cas: str | None = None,
    duration: int | None = None,
    per_species_agg: str = "median",
    min_species: int = 5,
) -> tuple[np.ndarray, np.ndarray, str]:
    """
    Returns species-level toxicity mean and sd arrays for the SSD, plus the CAS chosen.
    - df must contain columns: ['CAS','species','pred_mean','total_sd'] and (optionally) 'duration'
    - per_species_agg: 'median' or 'min' (regulatory-conservative)
    """
    work = df.copy()

    # Choose CAS if not provided: pick the chemical with most observations
    if cas is None:
        cas = work["CAS"].value_counts().idxmax()

    work = work[work["CAS"] == cas].copy()

    if duration is not None and "duration" in work.columns:
        # duration is categorical in your pipeline; cast safely to int for comparison
        with np.errstate(invalid="ignore"):
            dur_int = pd.to_numeric(work["duration"], errors="coerce").astype("Int64")
        work = work[dur_int == int(duration)]

    # Guardrails
    if work.empty:
        raise ValueError("No rows after filtering by CAS/duration.")
    required = {"species", "pred_mean", "total_sd"}
    missing = required.difference(work.columns)
    if missing:
        raise ValueError(f"df is missing columns: {missing}")

    # Aggregate to one value per species
    if per_species_agg == "median":
        agg_mu = work.groupby("species", observed=True)["pred_mean"].median()
        agg_sd = work.groupby("species", observed=True)["total_sd"].median()
    elif per_species_agg == "min":
        # choose the row with the lowest predicted toxicity per species
        idx = work.groupby("species", observed=True)["pred_mean"].idxmin()
        agg = work.loc[idx, ["species", "pred_mean", "total_sd"]].set_index("species")
        agg_mu, agg_sd = agg["pred_mean"], agg["total_sd"]
    else:
        raise ValueError("per_species_agg must be 'median' or 'min'.")

    mu = agg_mu.to_numpy(dtype=float)
    sd = agg_sd.to_numpy(dtype=float)

    # Clean up sds (clip and fill)
    sd = np.nan_to_num(sd, nan=0.0, posinf=0.0, neginf=0.0)
    sd = np.clip(sd, 0.0, None)
    sd[sd < 1e-8] = 1e-8  # ensure finite noise for sampling

    if mu.shape[0] < min_species:
        raise ValueError(f"Too few species ({mu.shape[0]}). Need at least {min_species} for an SSD.")

    return mu, sd, str(cas)


def ssd_with_bands(
    mu: np.ndarray,
    sd: np.ndarray,
    n_sims: int = 2000,
    ci: float = 0.90,
    percentile_grid: np.ndarray | None = None,
    rng: np.random.Generator | None = None,
):
    """
    Monte Carlo SSD:
      - For each simulation, draw one predictive sample per species:
            y_i ~ Normal(mu_i, sd_i^2)
      - For each percentile p (fraction of species affected), compute the
        quantile across species within each simulation.
      - Summarize across simulations to get median SSD and a pointwise
        credible band. Also return the HC5 posterior.
    """
    if rng is None:
        rng = np.random.default_rng(20250810)

    mu = np.asarray(mu, dtype=float).ravel()
    sd = np.asarray(sd, dtype=float).ravel()
    assert mu.shape == sd.shape and mu.ndim == 1

    if percentile_grid is None:
        percentile_grid = np.linspace(0.01, 0.99, 99)  # 1%..99%
    percentile_grid = np.asarray(percentile_grid, dtype=float).ravel()

    # Posterior predictive draws: shape (n_sims, S)
    draws = mu[None, :] + sd[None, :] * rng.standard_normal(size=(n_sims, mu.size))

    # For each simulation, quantiles across species (axis=1)
    # Result shape: (len(p), n_sims)
    q_sim = np.quantile(draws, percentile_grid, axis=1)

    lo_q = (1.0 - ci) / 2.0
    hi_q = 1.0 - lo_q

    # Summarize across simulations (axis=1)
    q_med = np.quantile(q_sim, 0.5, axis=1)
    q_lo  = np.quantile(q_sim, lo_q, axis=1)
    q_hi  = np.quantile(q_sim, hi_q, axis=1)

    # HC5 per simulation: 5th percentile across species for each sim
    hc5_sims = np.quantile(draws, 0.05, axis=1)
    hc5_med  = float(np.quantile(hc5_sims, 0.5))
    hc5_lo   = float(np.quantile(hc5_sims, lo_q))
    hc5_hi   = float(np.quantile(hc5_sims, hi_q))

    return {
        "p": percentile_grid,       # fraction of species affected
        "q_med": q_med,             # median concentration at each p
        "q_lo": q_lo,               # lower band
        "q_hi": q_hi,               # upper band
        "hc5_sims": hc5_sims,       # distribution of HC5
        "hc5_med": hc5_med,
        "hc5_lo": hc5_lo,
        "hc5_hi": hc5_hi,
        "ci": ci,
    }


def plot_ssd_with_bands(ssd, cas_label: str, units_label: str = "model scale", logx: bool = False):
    """
    Plot SSD median curve with pointwise credible band; vertical line for HC5.
    The x-axis is concentration, y-axis is fraction of species.
    """
    p   = np.asarray(ssd["p"]).ravel()
    med = np.asarray(ssd["q_med"]).ravel()
    lo  = np.asarray(ssd["q_lo"]).ravel()
    hi  = np.asarray(ssd["q_hi"]).ravel()
    ci  = float(ssd.get("ci", 0.90))

    # Ensure band ordering
    lo2 = np.minimum(lo, hi)
    hi2 = np.maximum(lo, hi)
    lo, hi = lo2, hi2

    fig, ax = plt.subplots(figsize=(8, 5))

    # Horizontal band: for each y=p, fill x in [lo, hi]
    ax.fill_betweenx(p, lo, hi, alpha=0.2, label=f"{int(ci*100)}% credible band")

    # Median SSD curve
    ax.plot(med, p, linewidth=2, label="Median SSD")

    # HC5 marker
    ax.axvline(ssd["hc5_med"], linewidth=1.5, linestyle=":", label=f"HC5 (median) = {ssd['hc5_med']:.3f}")

    ax.set_xlabel(f"Toxicity [{units_label}]")
    ax.set_ylabel("Fraction of species affected")
    ax.set_title(f"Species Sensitivity Distribution (SSD)\nChemical: {cas_label}")
    ax.set_ylim(0.0, 1.0)
    if logx:
        ax.set_xscale("log")
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig, ax


def plot_hc5_posterior(ssd, cas_label: str, units_label: str = "model scale"):
    """
    One chart: histogram of HC5 posterior (companion figure).
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(ssd["hc5_sims"], bins=40)
    ax.axvline(ssd["hc5_med"], linewidth=1.5, linestyle=":")
    ax.set_xlabel(f"HC5 [{units_label}]")
    ax.set_ylabel("Frequency")
    ax.set_title(f"HC5 posterior — {cas_label}\nmedian={ssd['hc5_med']:.3f}, "
                 f"90% CI=({ssd['hc5_lo']:.3f}, {ssd['hc5_hi']:.3f})")
    fig.tight_layout()
    return fig, ax


# ==== Example usage ====
# df must already exist with columns:
# ['CAS','species','pred_mean','total_sd', ... (optionally 'duration')]
CAS_EXAMPLE   = None          # or e.g. "50-00-0"
DURATION_HOURS = None         # e.g. 96 to restrict, else None
UNITS          = "log LC50 (centered)"  # change if you uncenter

mu, sd, chosen_cas = prepare_species_values(
    df,
    cas=CAS_EXAMPLE,
    duration=DURATION_HOURS,
    per_species_agg="median"   # or "min" for conservative
)

ssd = ssd_with_bands(mu, sd, n_sims=2000, ci=0.90)

print(f"CAS: {chosen_cas}")
print(f"HC5 (median) = {ssd['hc5_med']:.3f}; 90% CI = ({ssd['hc5_lo']:.3f}, {ssd['hc5_hi']:.3f})")

fig1, ax1 = plot_ssd_with_bands(ssd, chosen_cas, units_label=UNITS, logx=False)  # set True for log-x
fig2, ax2 = plot_hc5_posterior(ssd, chosen_cas, units_label=UNITS)


In [ ]:
results = pd.read_csv('outputs/A1_fp_grid.csv')

In [ ]:
results.head()

In [ ]:
results.sort_values(by = ['mean_rmse'])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

def plot_ssd_with_points_revised(mu: np.ndarray, sd: np.ndarray, cas_label: str, units_label: str = "model scale", ci: float = 0.90):
    """
    Plots a revised, cleaner Species Sensitivity Distribution (SSD) as individual points
    with their confidence bands. Visuals are optimized for dense plots.

    Args:
        mu (np.ndarray): Array of mean predictions for each species.
        sd (np.ndarray): Array of total standard deviations for each species.
        cas_label (str): The CAS number or name of the chemical for the plot title.
        units_label (str): The label for the x-axis units.
        ci (float): The confidence interval to display (e.g., 0.90 for 90%).
    """
    # Sort the mean predictions to form the SSD shape
    sorted_indices = np.argsort(mu)
    sorted_mu = mu[sorted_indices]
    sorted_sd = sd[sorted_indices]

    # Calculate the empirical cumulative probability for the y-axis
    n = len(sorted_mu)
    p = (np.arange(n) + 1) / (n + 1)

    # Calculate the error for the confidence interval
    z_score = norm.ppf(1 - (1 - ci) / 2)
    error = z_score * sorted_sd

    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot with optimized visual parameters
    ax.errorbar(
        x=sorted_mu,
        y=p,
        xerr=error,
        fmt='o',              # Point marker
        linestyle='None',     # No line connecting points
        markersize=3,         # Smaller markers
        alpha=0.4,            # Add transparency to see density
        elinewidth=0.8,       # --- Thinner error bar lines ---
        capsize=2,            # Smaller caps on the error bars
        capthick=0.8,         # Thinner cap lines
        label=f"Model Predictions with {int(ci*100)}% CI"
    )

    ax.set_xlabel(f"Toxicity [{units_label}]", fontsize=12)
    ax.set_ylabel("Fraction of Species Affected (Empirical)", fontsize=12)
    ax.set_title(f"Species Sensitivity Distribution (SSD)\nChemical: {cas_label}", fontsize=16, pad=15)
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig, ax

# ==== Example Usage with the Revised Plotting Function ====
# Ensure the 'df' DataFrame from the previous cells is available
CAS_EXAMPLE = "7758-98-7"  # Using the chemical from your image
UNITS = "log LC50 (centered)"

# Prepare the data as before
mu, sd, chosen_cas = prepare_species_values(
    df,
    cas=CAS_EXAMPLE,
    per_species_agg="median"
)

# --- Generate and display the improved plot ---
print(f"Generating revised SSD plot for {chosen_cas}.")
fig, ax = plot_ssd_with_points_revised(mu, sd, chosen_cas, units_label=UNITS, ci=0.8)
plt.show()